# Soft Actor-Critic (SAC)

https://spinningup.openai.com/en/latest/algorithms/sac.html

In [1]:
import torch
from tensordict.nn import TensorDictModule, TensorDictSequential, NormalParamExtractor
from torch import nn
from torchrl import logger
from torchrl.collectors import Collector
from torchrl.data import LazyTensorStorage, ReplayBuffer
from torchrl.envs import Compose, DoubleToFloat, GymEnv, StepCounter, TransformedEnv
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import (
    AdditiveGaussianModule,
    ProbabilisticActor,
    TanhNormal,
    ValueOperator,
)
from torchrl.objectives import SACLoss, SoftUpdate

/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:574: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  PUCT = functools.partial(PUCTScore, c=5)  # AlphaGo default value
/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:575: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB = functools.partial(UCBScore, c=math.sqrt(2))  # default from Auer et al. 2002
/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:576: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB1_TUNED = functools.partial(
/usr/local/Caskroom/mini

In [2]:
env = TransformedEnv(
    GymEnv("InvertedPendulum-v5"),
    Compose([DoubleToFloat(), StepCounter()]),
)

In [3]:
_ = env.set_seed(0)
_ = torch.manual_seed(0)

Use a stochastic policy, similar to that of PPO.

In [4]:
NUM_CELLS = 256

policy_net = nn.Sequential(
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(2),
    NormalParamExtractor(),
)

policy_module = ProbabilisticActor(
    TensorDictModule(policy_net, in_keys=["observation"], out_keys=["loc", "scale"]),
    in_keys=["loc", "scale"],
    spec=env.action_spec,
    safe=True,
    distribution_class=TanhNormal,
    distribution_kwargs={
        "low": env.action_spec.space.low,  # type: ignore
        "high": env.action_spec.space.high,  # type: ignore
    },
)

Use the same Q-value network as in DDPG and TD3;
the network is duplicated by the loss module.

In [5]:
class ActionValueNetwork(nn.Module):
    def __init__(self, observation_net: nn.Module, action_net: nn.Module):
        super().__init__()
        self.observation_net = observation_net
        self.action_net = action_net

    def forward(self, observation: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        xs = self.observation_net(observation)
        xs = nn.Tanh()(xs)
        xs = torch.cat([xs, action], dim=-1)
        xs = self.action_net(xs)

        return xs


qvalue_net = ActionValueNetwork(
    observation_net=nn.Sequential(
        nn.LazyLinear(NUM_CELLS), nn.Tanh(), nn.LazyLinear(NUM_CELLS)
    ),
    action_net=nn.Sequential(nn.LazyLinear(NUM_CELLS), nn.Tanh(), nn.LazyLinear(1)),
)

qvalue_module = ValueOperator(qvalue_net, in_keys=["observation", "action"])

In [6]:
_ = qvalue_module(env.rollout(10, policy_module))  # initialize lazy layers

In [7]:
LEARNING_RATE = 1e-3

loss_module = SACLoss(
    actor_network=policy_module,
    qvalue_network=qvalue_module,
    action_spec=env.action_spec,
    num_qvalue_nets=2,  # use two Q-value networks (original "twin")
    alpha_init=1.0,  # initial entropy multiplier
    target_entropy=-1.0,  # optimize alpha to reach the target
    delay_actor=False,  # (default value) actions come from current policy
)

updater = SoftUpdate(loss_module, eps=0.99)

optimizer = torch.optim.Adam(loss_module.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 100_000)

Add Gaussian noise to the policy to facilitate exploration early on.

In [8]:
exploration_module = AdditiveGaussianModule(
    spec=env.action_spec,
    annealing_num_steps=100_000,
    safe=True,
)

exploration_policy = TensorDictSequential([policy_module, exploration_module])

Same collector and replay buffer setup as in DDPG/TD3.

In [9]:
FRAMES_PER_BATCH = 100
INIT_RANDOM_FRAMES = 5000

collector = Collector(
    env,
    policy=exploration_policy,
    frames_per_batch=FRAMES_PER_BATCH,
    init_random_frames=INIT_RANDOM_FRAMES,
    total_frames=-1,
)

buffer = ReplayBuffer(storage=LazyTensorStorage(max_size=100_000))

Evaluate the stochastic policy in a deterministic manner (same as for PPO).

In [10]:
@torch.no_grad()
def evaluate_policy(env: TransformedEnv, policy_module: ProbabilisticActor):
    env.reset()

    with set_exploration_type(ExplorationType.DETERMINISTIC):
        rollout = env.rollout(1000, policy_module)

    env.reset()

    return rollout["next", "step_count"].max().item()

Same training loops as in DQN and DDPG, except that we also optimize the entropy multiplier `alpha` (which is not necessary).

In [11]:
OPTIM_STEPS = 10
BATCH_SIZE = 128

step_count = 0
episode_count = 0


for idx, data in enumerate(collector, start=1):
    buffer.extend(data)

    step_count += data.numel()
    episode_count += data["next", "done"].sum()

    if len(buffer) < collector.init_random_frames:
        continue

    max_steps = evaluate_policy(env, policy_module)

    if idx % 10 == 0:
        logger.info(f"[{idx:>3}] steps: {max_steps:>3}")

    if max_steps > 200:
        break

    for optim_step in range(OPTIM_STEPS):
        data_batch = buffer.sample(BATCH_SIZE)
        batch_loss = loss_module(data_batch)

        loss = (
            batch_loss["loss_actor"]
            + batch_loss["loss_qvalue"]
            + batch_loss["loss_alpha"]
        )
        loss.backward()

        nn.utils.clip_grad_norm_(loss_module.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

        updater.step()
        scheduler.step()

    exploration_module.step(data.numel())

logger.info(f"solved after {step_count} steps, {episode_count} episodes")

2026-02-16 14:32:32,526 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([100000]) shape [END]
2026-02-16 14:32:37,181 [torchrl][INFO]    [ 50] steps:   5 [END]
2026-02-16 14:32:40,605 [torchrl][INFO]    [ 60] steps:   7 [END]
2026-02-16 14:32:44,071 [torchrl][INFO]    [ 70] steps:  70 [END]
2026-02-16 14:32:48,065 [torchrl][INFO]    [ 80] steps:  73 [END]
2026-02-16 14:32:52,442 [torchrl][INFO]    [ 90] steps:  73 [END]
2026-02-16 14:32:57,017 [torchrl][INFO]    [100] steps:  85 [END]
2026-02-16 14:33:01,840 [torchrl][INFO]    [110] steps:  80 [END]
2026-02-16 14:33:06,790 [torchrl][INFO]    [120] steps: 164 [END]
2026-02-16 14:33:11,878 [torchrl][INFO]    [130] steps:  80 [END]
2026-02-16 14:33:17,534 [torchrl][INFO]    [140] steps: 110 [END]
2026-02-16 14:33:23,835 [torchrl][INFO]    [150] steps:  78 [END]
2026-02-16 14:33:29,796 [torchrl][INFO]    [160] steps:  80 [END]
2026-02-16 14:33:31,357 [torchrl][INFO]    solved after 16200 steps, 1318 episodes [END]
